In [2]:
# Test the plane wave again, but check intermediate values
import torch
import numpy as np
from helmholtz import Helmholtz_Loss
import scipy
from scipy import io

k = 5.5
l = 3.0 / 31
helmholtz_fn = Helmholtz_Loss(k, l)

# Plane wave: u = cos(kx)
x = torch.zeros(1, 8, 32, 32)
grid = torch.linspace(0, 3.0, 32)

for i in range(32):
    for j in range(32):
        pos_x = grid[j]
        x[0, 0, i, j] = np.cos(k * pos_x)        # u
        x[0, 1, i, j] = -k * np.sin(k * pos_x)   # du/dx
        x[0, 2, i, j] = 0                         # du/dy
        x[0, 3, i, j] = 0                         # d²u/dxdy

print(f"Plane wave pressure max: {x[0,0].abs().max()}")
print(f"Plane wave du/dx max: {x[0,1].abs().max()}")
print(f"Plane wave loss: {helmholtz_fn(x).item()}")

# Now load MATLAB data and DON'T transpose
mat_data = scipy.io.loadmat('/Users/leifefrancisco/Documents/Workspace/PI_CNN_V2/PI_CNN_V3a/V3_a_a_0_001/data/room_acoustic_data_room_0005.mat')
u_grid = mat_data['u_grid']  # No transpose

# Compute derivatives with original axis convention
du_dx = np.gradient(u_grid.real, l, axis=1)
du_dy = np.gradient(u_grid.real, l, axis=0)
du_dxdy = np.gradient(du_dx, l, axis=0)

du_dx_imag = np.gradient(u_grid.imag, l, axis=1)
du_dy_imag = np.gradient(u_grid.imag, l, axis=0)
du_dxdy_imag = np.gradient(du_dx_imag, l, axis=0)

print(f"\nMATLAB pressure max: {np.abs(u_grid).max()}")
print(f"MATLAB du/dx max: {np.abs(du_dx).max()}")
print(f"MATLAB du/dy max: {np.abs(du_dy).max()}")

# Check if maybe the issue is float32 vs float64
helmholtz_input = torch.zeros(1, 8, 32, 32, dtype=torch.float64)
helmholtz_input[0, 0] = torch.from_numpy(u_grid.real)
helmholtz_input[0, 1] = torch.from_numpy(du_dx)
helmholtz_input[0, 2] = torch.from_numpy(du_dy)
helmholtz_input[0, 3] = torch.from_numpy(du_dxdy)
helmholtz_input[0, 4] = torch.from_numpy(u_grid.imag)
helmholtz_input[0, 5] = torch.from_numpy(du_dx_imag)
helmholtz_input[0, 6] = torch.from_numpy(du_dy_imag)
helmholtz_input[0, 7] = torch.from_numpy(du_dxdy_imag)

# Convert Helmholtz buffers to float64 too
helmholtz_fn = helmholtz_fn.double()
loss = helmholtz_fn(helmholtz_input)
print(f"MATLAB loss (float64): {loss.item()}")

Plane wave pressure max: 1.0
Plane wave du/dx max: 5.498144149780273
Plane wave loss: 596.5083618164062

MATLAB pressure max: 0.9999999999999999
MATLAB du/dx max: 5.402650920554113
MATLAB du/dy max: 4.872556968065762
MATLAB loss (float64): 12.923234359190115


/var/folders/3k/4qh7ydgj17b9ps65dp9mswz00000gn/T/ipykernel_1373/2067215821.py:19: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x[0, 0, i, j] = np.cos(k * pos_x)        # u
/var/folders/3k/4qh7ydgj17b9ps65dp9mswz00000gn/T/ipykernel_1373/2067215821.py:20: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x[0, 1, i, j] = -k * np.sin(k * pos_x)   # du/dx


In [3]:
print(f"Plane wave d²u/dxdy max: 0")
print(f"MATLAB d²u/dxdy max: {np.abs(du_dxdy).max()}")
print(f"MATLAB d²u/dxdy_imag max: {np.abs(du_dxdy_imag).max()}")

Plane wave d²u/dxdy max: 0
MATLAB d²u/dxdy max: 30.146937210233336
MATLAB d²u/dxdy_imag max: 14.200433639562682


In [4]:
# 2D plane wave: u = cos(kx*x + ky*y) where kx² + ky² = k²
kx = k / np.sqrt(2)
ky = k / np.sqrt(2)

x = torch.zeros(1, 8, 32, 32)
grid = torch.linspace(0, 3.0, 32)

for i in range(32):
    for j in range(32):
        pos_x = grid[j]
        pos_y = grid[i]
        
        x[0, 0, i, j] = np.cos(kx * pos_x + ky * pos_y)           # u
        x[0, 1, i, j] = -kx * np.sin(kx * pos_x + ky * pos_y)     # du/dx
        x[0, 2, i, j] = -ky * np.sin(kx * pos_x + ky * pos_y)     # du/dy
        x[0, 3, i, j] = -kx * ky * np.cos(kx * pos_x + ky * pos_y) # d²u/dxdy

print(f"2D plane wave loss: {helmholtz_fn(x).item()}")

/var/folders/3k/4qh7ydgj17b9ps65dp9mswz00000gn/T/ipykernel_1373/2053497609.py:13: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x[0, 0, i, j] = np.cos(kx * pos_x + ky * pos_y)           # u
/var/folders/3k/4qh7ydgj17b9ps65dp9mswz00000gn/T/ipykernel_1373/2053497609.py:14: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x[0, 1, i, j] = -kx * np.sin(kx * pos_x + ky * pos_y)     # du/dx
/var/folders/3k/4qh7ydgj17b9ps65dp9mswz00000gn/T/ipykernel_1373/2053497609.py:15: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  x[0, 2, i, j] = -ky * np.sin(kx * pos_x + ky * pos_y)     # du/dy
/var/folders/3k/4qh7ydgj17b9ps65dp9mswz00000gn/T/ipykernel_1373/2053497609.py:16: DeprecationWarning: __array_wrap__ must accept context and retur

RuntimeError: expected scalar type Double but found Float

In [ ]:
# 2D plane wave: u = cos(kx*x + ky*y) where kx² + ky² = k²
kx = k / np.sqrt(2)
ky = k / np.sqrt(2)

x = torch.zeros(1, 8, 32, 32)
grid = torch.linspace(0, 3.0, 32)

for i in range(32):
    for j in range(32):
        pos_x = grid[j]
        pos_y = grid[i]
        
        x[0, 0, i, j] = np.cos(kx * pos_x + ky * pos_y)           # u
        x[0, 1, i, j] = -kx * np.sin(kx * pos_x + ky * pos_y)     # du/dx
        x[0, 2, i, j] = -ky * np.sin(kx * pos_x + ky * pos_y)     # du/dy
        x[0, 3, i, j] = -kx * ky * np.cos(kx * pos_x + ky * pos_y) # d²u/dxdy

# Create fresh float32 Helmholtz loss
helmholtz_fn_float = Helmholtz_Loss(k, l)
print(f"2D plane wave loss: {helmholtz_fn_float(x).item()}")

NameError: name 'k' is not defined